# A. AIS Data Prep

**Overview.** First stage of the Pacific ship-emissions pipeline. Reads raw satellite/terrestrial AIS vessel-position messages, restricts them to the Pacific bounding box and the EEZs of the Pacific Island countries, matches each vessel to the IHS ship register (to obtain an `LRIMOShipNo` / `vessel_id`), resamples positions to one record per vessel per hour, and tags every hourly position with geospatial flags — inside a country EEZ, within 10 km of the coast, and inside an IMF port boundary. The result is the hourly AIS activity panel that every downstream emissions notebook consumes.

**Series.** This is notebook **A** of a five-part series (A–E) that estimates ship greenhouse-gas and air-pollutant emissions in Pacific Island EEZs following the Fourth IMO GHG Study 2020 bottom-up method:
- **A. AIS Data Prep** — build the hourly vessel-position panel, tagged with EEZ/coast/port and IHS vessel IDs *(this notebook)*
- **B. IHS Specs** — per-vessel technical specifications, correction factors, and emission factors from the IHS ship database
- **C. Emissions wIHS** — activity-based emissions for IHS-matched vessels
- **D. Emissions woIHS** — emissions for vessels without an IHS match, using a median-per-type model derived from C
- **E. Emissions Statistics** — monthly country / vessel-type / fishing aggregations and dashboard CSVs

**Inputs.**
- Raw AIS messages, read via the platform `ais` library (`af.get_ais`).
- IHS ship tables (`ShipData.CSV`, `tblNameHistory.CSV`) via `af.read_ihs_table` — IHS version pinned by `ihs_version` (run used `20260202`).
- `imf_port_boundary.pkl` — IMF port boundary polygons.
- `geoboundary2.pkl` — EEZ and 10 km coastal-buffer polygons per country.
- All paths are under the platform S3 root `"s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"`.

**Output.** Hourly AIS parquet partitioned by `year`/`month` at `…/emissions/Pacific/hourly_ais_v2/`, one row per `vessel_id`-hour with first/last/mean/max/min speed and draught, position, and the `eez_flag` / `coast_flag` / `port_flag` / `Country` tags.

**Requirements.**
- Runs on the **UN Global Platform** (UN Big Data hub) — an account is required: https://datalab.officialstatistics.org
- Kernel: `pyspark3.5 ais2.9`. Uses the platform's `ais` library (AIS Task Team) on a PySpark cluster with **Apache Sedona** for the spatial joins (`ST_Point`, `ST_Contains`, `ST_GeomFromWKT`); plus `geopandas`/`shapely` for boundary prep.
- Note on coordinates: longitudes are transposed to 0–360 (`longitude + 360` where ≤ 0) so the Pacific antimeridian region is contiguous.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
from shapely.geometry import Polygon, Point
import shapely as s

In [2]:
from pyspark.sql import SparkSession
from ais import functions as af
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()


In [3]:
wb_path = "s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"
project_path = wb_path + "emissions/Pacific/"

## IMF Port Boundaries

_Loads the IMF port-boundary polygons for the 14 Pacific countries and transposes them to 0–360° longitude (so they stay contiguous across the antimeridian); flags them as port areas._


In [4]:
def pacific_transpose(gdf_series):
    ref = gdf_series.explode(index_parts=True)
    coord = ref.get_coordinates().reset_index()
    coord['x_t'] = coord['x'] + 360*(coord['x'] <= 0)
    basis = coord.groupby(['level_0','level_1'])['x'].apply(lambda x: sum(x<=0))
    trans = gpd.GeoDataFrame(coord.groupby(['level_0','level_1'])[['x_t','y']].apply(lambda x: Polygon(x.values.tolist())).reset_index(),
                        crs=4326,
                        geometry=0).dissolve(['level_0','level_1'])[0]

    fin = gpd.GeoDataFrame(pd.concat([ref,basis,trans], axis=1))
    fin.columns = ['orig','basis','trans']
    fin['fin'] = np.where(fin['basis'] ==0, fin['orig'], fin['trans'])
    fin = fin.set_geometry("fin").set_crs(4326)
    return fin.reset_index().dissolve(["level_0"])['fin']

In [5]:
imf = pd.read_pickle(wb_path + "imf_port_boundary.pkl")

country_list = ['Samoa', 'Tonga', 'Kiribati','Fiji', 'Vanuatu', 'Nauru',
       'Micronesia', 'Palau',
       'Solomon Islands', 'Marshall Islands', 'Tuvalu',
       'Papua New Guinea', 'Cook Islands', 'Niue']


imf = imf[imf['Country'].isin(country_list)].drop_duplicates(subset=['Port_name','Country'])[['Country','port_boundary','Port_name']]



imf['port_boundary_t'] = pacific_transpose(imf['port_boundary'])
imf['port_flag']  = True

Found credentials from IAM Role: eksctl-sparky-mc-sparkface-nodegr-NodeInstanceRole-15Y58CLME2WUS


In [6]:
imf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Int64Index: 51 entries, 297 to 1441
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Country          51 non-null     object  
 1   port_boundary    51 non-null     geometry
 2   Port_name        51 non-null     object  
 3   port_boundary_t  51 non-null     geometry
 4   port_flag        51 non-null     bool    
dtypes: bool(1), geometry(2), object(2)
memory usage: 2.0+ KB


## EEZ and Coast

_Loads each country's EEZ polygon and 10 km coastal buffer (also transposed) used to tag vessel positions._


In [7]:
gdf = pd.read_pickle(project_path + "geoboundary2.pkl")
gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   ISO3          17 non-null     object  
 1   Country       17 non-null     object  
 2   eez_t         17 non-null     geometry
 3   coast_10km_t  17 non-null     object  
 4   eez_flag      17 non-null     bool    
 5   coast_flag    17 non-null     bool    
dtypes: bool(2), geometry(1), object(3)
memory usage: 706.0+ bytes


# AIS

_Defines the core building blocks: match AIS vessels to IHS, attach EEZ/coast/port flags via Sedona spatial joins, resample raw positions to hourly per-vessel records, and the `run()` driver that writes the output._


In [8]:
def attach_ihs(ais, ihs_version):  
    ais_vessels = ais.select("imo","mmsi","vessel_name","dt_pos_utc").dropDuplicates(["imo","mmsi","vessel_name"])

    ihs = af.read_ihs_table(spark, "ShipData.CSV", version=ihs_version) \
    .select("LRIMOShipNo","MaritimeMobileServiceIdentityMMSINumber","ShipName","ExName")
    
    
    ihs_ais_vessel = af.match_ais_ihs(ais_vessels, 
                           ihs, 
                           name_similarity_measure =  "Indel", 
                           name_similarity_maxscore = 0.4, 
                           ) \
    .select("LRIMOShipNo","imo","mmsi","vessel_name","ihs_matchtype") \
    .withColumn("ihs_version", F.lit(ihs_version))
    
    rem_for_matching = ais_vessels.join(ihs_ais_vessel.select("imo","mmsi","vessel_name"),
                       on=["imo","mmsi","vessel_name"],
                       how="left_anti")

    ihs_name_hist = af.read_ihs_table(spark, "tblNameHistory.CSV", version=ihs_version)
    
    ihs_ais_namehist = af.ais_ihs_name_hist_match(rem_for_matching, 
                                              ihs_name_hist,
                                              name_similarity_measure =  "Indel", 
                           name_similarity_maxscore = 0.4, 
                           )
    
    ihs_ais = ihs_ais_vessel.select("LRIMOShipNo","imo","mmsi","vessel_name","ihs_matchtype").unionByName(
        ihs_ais_namehist.select(F.col("LRNO").alias("LRIMOShipNo"),"imo","mmsi","vessel_name","ihs_matchtype")
    )
    
    ais_ihs_all = ais.na.fill({'imo':0,"vessel_name":""}).join(
        ihs_ais.na.fill({'imo':0,"vessel_name":""}),
         on=['imo','mmsi','vessel_name'],
         how='left'
        ) \
    .withColumn("vessel_id", F.when(F.col("LRIMOShipNo").isNotNull(), F.concat(F.lit("imo-"),F.col("LRIMOShipNo").cast(StringType()))) \
                                 .otherwise(F.concat(F.lit("mmsi"), F.col("mmsi").cast(StringType()))))
    return ais_ihs_all

In [9]:
def attach_geo(ais, sdf_eez, sdf_port):
    ais_point = ais \
    .withColumnRenamed("latitude_f","latitude").withColumnRenamed("longitude_f","longitude") \
    .withColumn("longitude_t",F.when(F.col("longitude") <= 0, F.col("longitude") + 360).otherwise(F.col("longitude"))) \
    .withColumn("point", F.expr("ST_Point(longitude_t,latitude)"))
    

    ais_eez = ais_point.alias("pointDf") \
    .join(F.broadcast(sdf_eez.select("Country","ISO3","eez_t_geom", F.lit(True).alias("eez_flag")).alias("eezDf")), 
          F.expr("ST_Contains(eezDf.eez_t_geom, pointDf.point)"),
          how="left") \
    .join(F.broadcast(sdf_eez.select("coast_10km_t_geom", F.lit(True).alias("coast_flag")).alias("coastDf")),
          F.expr("ST_Contains(coastDf.coast_10km_t_geom, pointDf.point)"),
          how="left") \
    .join(F.broadcast(sdf_port.select("Port_name","port_t_geom", F.lit(True).alias("port_flag")).alias("portDf")),
          F.expr("ST_Contains(portDf.port_t_geom, pointDf.point)"),
          how="left")
    
    return ais_eez.drop("eez_t_geom","coast_10km_t_geom","port_t_geom","point") \
                    .na.fill(False, ["eez_flag","coast_flag","port_flag"])

In [10]:
from itertools import chain
def ais_hourly_prep(sdf):
    #re sample per hour    
    sdf_wcols = sdf.withColumn("date",F.to_date("dt_pos_utc")) \
        .withColumn("year",F.year("dt_pos_utc")) \
        .withColumn("month",F.month("dt_pos_utc")) \
        .withColumn("hour", F.hour("dt_pos_utc")) \
        .withColumn("sog_navi", F.when(F.col("nav_status_code").isin([0,8]), F.col("sog")))

        ###note this is incorrect for 11:30 pm to 12 because sets hour to 0, and therefore make it the same as 12:00AM, if set to 0, should have added 1 day more
            #     .withColumn("minute", F.minute("dt_pos_utc")) \
        # .withColumn("hour", F.when(F.col("minute") < 30, F.col("hour_floor")) \
        #                     .when((F.col("minute") >= 30) & (F.col("hour_floor") == 23), 0) \
        #                     .otherwise(F.col("hour_floor") + 1)
        #                          ) \

    
    groupby_cols = ['vessel_id']
    static_cols = ['imo','mmsi','LRIMOShipNo',"ihs_matchtype",'vessel_name','length','width',"vessel_type"]
    first_cols =  ['longitude','latitude','sog','draught','destination','nav_status']
    last_cols =   ['longitude','latitude','sog','draught','destination','nav_status']
    stats_cols = ['sog','draught','sog_navi']


    func_static_cols = [F.first(col).alias(col) for col in static_cols]
    func_f_cols = [F.min_by(col, "dt_pos_utc").alias(col+"_f") for col in first_cols]    
    func_l_cols = [F.max_by(col, "dt_pos_utc").alias(col+"_l") for col in last_cols]
    func_stats_cols = list(chain.from_iterable([[F.mean(col).alias(f"{col}_mean"), F.max(col).alias(f"{col}_max"), F.min(col).alias(f"{col}_min")] for col in stats_cols]))
    
    sdf_agg = sdf_wcols.groupby(*groupby_cols,"year","month","date","hour").agg(
        *func_static_cols,
        *func_f_cols,
        *func_l_cols,
        *func_stats_cols,
        F.count("mmsi").alias("count_ais")
    )

    return sdf_agg

In [11]:
columns = ['imo','mmsi','vessel_name','vessel_type','sog','draught','length','width','nav_status','nav_status_code',
           'destination','dt_pos_utc','longitude','latitude'
           ]

minx, maxx, miny, maxy = 120.9710, 236.1185, -25, 20

def run(start_date, end_date, ihs_version,mode="append", save_path = project_path+"hourly_ais_v2/"):
    
    sdf = af.get_ais(spark, start_date, end_date=end_date, columns=columns) \
    .withColumn("longitude_t", F.col("longitude") + F.when(F.col("longitude") <= 0, 360).otherwise(0)) \
    .filter(F.col("latitude").between(miny, maxy) & F.col("longitude_t").between(minx, maxx)) \
    .filter(F.col("mmsi").between(200000000,799999999)) \
    .filter((F.year("dt_pos_utc") == start_date.year) & (F.month("dt_pos_utc")==start_date.month))
    
    sdf_ihs = attach_ihs(sdf, ihs_version)
    sdf_hourly = ais_hourly_prep(sdf_ihs)
    sdf_geo = attach_geo(sdf_hourly, sdf_eez_coast,sdf_port)
    
    sdf_geo.repartition("year","month").write.mode(mode).partitionBy("year","month").parquet(save_path)
    
    # sdf_geo = spark.read.parquet(project_path+f"hourly_ais/year={year}/month={month}/")
    # print(sdf_geo.groupBy("Country").count())
    return sdf_geo

In [12]:
af.get_ihs_versions(spark)

,version,path,from_date,to_date
0,20260202,latest,2018-01-01,2026-02-25


In [13]:
ihs_version = "latest"

In [14]:
#Spark DataFrame port
sdf_port = spark.createDataFrame(imf[['Port_name','port_boundary_t']].astype(str)) \
.withColumn("port_t_geom", F.expr("ST_GeomFromWKT(port_boundary_t)")).cache()

In [15]:
sdf_eez_coast = spark.createDataFrame(gdf[['Country','ISO3','eez_t','coast_10km_t']].astype(str)) \
.withColumn("coast_10km_t_geom", F.expr("ST_GeomFromWKT(coast_10km_t)")) \
.withColumn("eez_t_geom",F.expr("ST_GeomFromWKT(eez_t)")).cache()

## Monthly

_Single-month test run (Feb 2024) with sanity-check counts by country, EEZ/coast/port, and lat/long bounds._


In [16]:
start_date = "2024-02-01"
end_date = "2024-02-29"

In [17]:
sdf = af.get_ais(spark, start_date, end_date=end_date, columns=columns) \
.withColumn("longitude_t", F.col("longitude") + F.when(F.col("longitude") <= 0, 360).otherwise(0)) \
.filter(F.col("latitude").between(miny, maxy) & F.col("longitude_t").between(minx, maxx)) \
.filter(F.col("mmsi").between(200000000,799999999))

sdf_ihs = attach_ihs(sdf, ihs_version)
sdf_hourly = ais_hourly_prep(sdf_ihs)
sdf_geo = attach_geo(sdf_hourly, sdf_eez_coast,sdf_port)

In [19]:
sdf_geo.repartition("year","month").write.mode("overwrite").partitionBy("year","month").parquet(wb_path+"temp/emissions")

In [20]:
sdf_geo =spark.read.parquet(wb_path+"temp/emissions") \
.withColumn("longitude_t",F.when(F.col("longitude") <= 0, F.col("longitude") + 360).otherwise(F.col("longitude"))) 

In [21]:
sdf_geo.count() #2465978

2465978

In [22]:
sdf_geo.filter(F.col("eez_flag")).count()

626157

In [23]:
sdf_geo.filter(F.col("coast_flag")).count()

193501

In [24]:
sdf_geo.filter(F.col("port_flag")).count()

113613

In [25]:
#seems okay, larger buffer same with S&P
sdf_geo.groupBy("Country").count().show()

+----------------+-------+
|         Country|  count|
+----------------+-------+
|   Phoenix Group|  14691|
|         Tokelau|   1024|
|           Tonga|   6686|
|            Fiji|  69279|
|           Palau|  13373|
|            NULL|1839821|
|      Line Group|   7311|
|Marshall Islands|  45153|
|          Tuvalu|  12687|
|           Nauru|   5546|
|           Samoa|  12019|
|    Cook Islands|  27809|
|Papua New Guinea| 213944|
| Gilbert Islands|  26826|
| Solomon Islands|  47921|
|      Micronesia|  81723|
|         Vanuatu|  39301|
|            Niue|    864|
+----------------+-------+



In [26]:
sdf_geo.select(F.min("latitude"),
           F.max("latitude"),
           F.min("longitude_t"),
           F.max("longitude_t")).show()

+-------------+-------------+----------------+----------------+
|min(latitude)|max(latitude)|min(longitude_t)|max(longitude_t)|
+-------------+-------------+----------------+----------------+
|        -25.0|         20.0|         120.971|       236.11848|
+-------------+-------------+----------------+----------------+



In [27]:
sdf_geo.groupBy("Country").agg(F.countDistinct("mmsi"),
                               F.countDistinct("imo")).show()

+----------------+--------------------+-------------------+
|         Country|count(DISTINCT mmsi)|count(DISTINCT imo)|
+----------------+--------------------+-------------------+
|   Phoenix Group|                 121|                 59|
|         Tokelau|                  36|                 24|
|           Tonga|                  78|                 51|
|            Fiji|                 338|                133|
|           Palau|                 431|                403|
|            NULL|               11307|               5493|
|      Line Group|                 145|                103|
|Marshall Islands|                 283|                193|
|          Tuvalu|                 107|                 80|
|           Nauru|                 103|                 77|
|           Samoa|                  73|                 39|
|    Cook Islands|                 164|                 77|
|Papua New Guinea|                1668|               1388|
| Gilbert Islands|                 168| 

## Loop

_Production loop: runs the pipeline month by month over a date range and appends the hourly output to `hourly_ais_v2/`._


In [14]:
spark.read.parquet(project_path+"hourly_ais_v2/year=2025/").select(F.max("date")).show()

+----------+
| max(date)|
+----------+
|2025-11-30|
+----------+



In [16]:
start_date_list = pd.date_range("2025-11-01", "2026-01-31", freq="MS")
end_date_list = pd.date_range("2025-11-01", "2026-01-31", freq="M")

In [20]:
i = 0
start_date = start_date_list[i]
end_date = end_date_list[i]

sdf = af.get_ais(spark, start_date, end_date=end_date, columns=columns) \
.withColumn("longitude_t", F.col("longitude") + F.when(F.col("longitude") <= 0, 360).otherwise(0)) \
.filter(F.col("latitude").between(miny, maxy) & F.col("longitude_t").between(minx, maxx)) \
.filter(F.col("mmsi").between(200000000,799999999)) \
.filter((F.year("dt_pos_utc") == start_date.year) & (F.month("dt_pos_utc")==start_date.month))

sdf_ihs = attach_ihs(sdf, ihs_version)
sdf_hourly = ais_hourly_prep(sdf_ihs)
sdf_geo = attach_geo(sdf_hourly, sdf_eez_coast,sdf_port)

In [39]:
sdf.count()

34241487

In [41]:
sdf.count()

34221120

In [22]:
# sdf_geo.repartition("year","month").write.mode("overwrite").partitionBy("year","month").parquet(wb_path+"emissions/temp/hourly_ais_v2/")
# 7 mins

In [ ]:
savepath =  wb_path+"emissions/temp/hourly_ais_v2/"
for i in range(1, len(start_date_list)):
    start_date = start_date_list[i]
    end_date = end_date_list[i]
    print(start_date)
    run(start_date, end_date, ihs_version,mode="append", save_path = savepath)

2025-12-01 00:00:00
2026-01-01 00:00:00
